In [1]:
"""
create_embeddings.ipynb forService 2 (Semantic Search).

For some reason I cannot resolve the paths in a .py file, hence the notebook

This script prepares your local ChromaDB collection. 
It reads a CSV of ArXiv-like paper metadata, generates embeddings using OpenAI, and stores
them in a persistent Chroma collection.

Should be run once before launching the chatbot:
    python create_embeddings.ipynb
"""

import pandas as pd
import chromadb
from langchain_openai import OpenAIEmbeddings
from tqdm import tqdm
import os
from dotenv import load_dotenv


load_dotenv("../.env")
load_dotenv("../.secrets")

# === Configuration ===
CSV_PATH = "./data/arXiv_scientific_dataset.csv"  # under 40MB
CHROMA_PATH = "./chromadb_store"
COLLECTION_NAME = "arxiv_metadata"

# === Load dataset ===
print(f"Loading dataset from {CSV_PATH} ...")
df = pd.read_csv(CSV_PATH)

# Expecting columns like: id, title, authors, year, primary_category, summary
required_cols = {"id", "title", "authors", "published_date", "category", "summary"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in dataset: {missing}")

# Clean / normalize data
df = df.dropna(subset=["summary", "title"])
df["id"] = df["id"].astype(str)
print(f"Loaded {len(df)} papers.")

# === Initialize ChromaDB client ===
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(COLLECTION_NAME)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key = os.getenv("OPENAI_API_KEY"))

# === Prepare data for embedding ===
ids = df["id"].tolist()
documents = df["summary"].tolist()
metadatas = df[["title", "authors", "published_date", "category"]].to_dict("records")

# === Chunk and embed ===
print(f"Creating and storing embeddings in {CHROMA_PATH} ...")
batch_size = 100
for i in tqdm(range(0, len(df), batch_size)):
    batch_ids = ids[i:i + batch_size]
    batch_docs = documents[i:i + batch_size]
    batch_meta = metadatas[i:i + batch_size]
    collection.upsert(ids=batch_ids, documents=batch_docs, metadatas=batch_meta)

print(f"Finished embedding {len(df)} papers.")
print(f"Collection '{COLLECTION_NAME}' is ready for semantic search.")


Loading dataset from ./data/arXiv_scientific_dataset.csv ...


C:\Users\Ramin\AppData\Local\Temp\ipykernel_29928\420977776.py:32: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


Loaded 14999 papers.
Creating and storing embeddings in ./chromadb_store ...


100%|██████████| 150/150 [24:23<00:00,  9.76s/it]

Finished embedding 14999 papers.
Collection 'arxiv_metadata' is ready for semantic search.
